# Pre-tokenization Cache for Retriever Training

In [1]:
import json
import os
import random
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from urllib.error import HTTPError, URLError
from urllib.request import urlretrieve

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


def detect_runtime() -> str:
    if os.environ.get("KAGGLE_URL_BASE") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"

RUNTIME = detect_runtime()

DATA_FILE_NAME = "training_data_v1.jsonl"
OUTPUT_FILE_NAME = "pretokenized_retriever_v1.pt"

@dataclass
class PreTokenizationConfig:
    model_name: str = "vinai/phobert-base"
    max_q_len: int = 96
    max_c_len: int = 256
    test_size: float = 0.30
    val_ratio_in_temp: float = 0.50
    output_path: Optional[str] = None
    data_file_name: str = DATA_FILE_NAME
    use_github_fallback: bool = True
    github_repo: str = "" 
    github_branch: str = "main"
    github_data_dir: str = "data"
    force_refresh_github_data: bool = False


cfg = PreTokenizationConfig()

if cfg.output_path is None:
    if RUNTIME == "kaggle":
        cfg.output_path = "/kaggle/working/data/" + OUTPUT_FILE_NAME
    elif RUNTIME == "colab":
        cfg.output_path = "/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1/data/" + OUTPUT_FILE_NAME
    else:
        cfg.output_path = "/data/" + OUTPUT_FILE_NAME

print(f"Runtime: {RUNTIME}")
print(cfg)

e:\AI\NLP\Legal-question-answer-v1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Runtime: local
PreTokenizationConfig(model_name='vinai/phobert-base', max_q_len=96, max_c_len=256, test_size=0.3, val_ratio_in_temp=0.5, output_path='/data/pretokenized_retriever_v1.pt', data_file_name='training_data_v1.jsonl', use_github_fallback=True, github_repo='', github_branch='main', github_data_dir='data', force_refresh_github_data=False)


In [2]:
repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/working/Legal-question-answer-v1"),
    Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1"),
]

repo_root = None
for candidate in repo_candidates:
    if (candidate / "utils").exists():
        repo_root = candidate
        break

if repo_root is None:
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

# Force local output to project data folder instead of filesystem root /data.
if RUNTIME == "local":
    cfg.output_path = str((repo_root / "data" / OUTPUT_FILE_NAME).resolve())

from utils.retriever_data_utils import build_training_pairs
from utils.get_jsonl_data import get_jsonl_data

def build_github_raw_url(repo: str, branch: str, rel_path: str) -> str:
    rel_path = rel_path.strip("/")
    return f"https://raw.githubusercontent.com/{repo}/{branch}/{rel_path}"


def download_data_from_github(
    file_name: str,
    repo: str,
    branch: str,
    github_data_dir: str,
    local_data_dir: Path,
    force_refresh: bool = False,
 ) -> Path:
    local_data_dir.mkdir(parents=True, exist_ok=True)
    target = local_data_dir / file_name

    if target.exists() and not force_refresh:
        return target

    rel_path = f"{github_data_dir.strip('/')}/{file_name}"
    url = build_github_raw_url(repo, branch, rel_path)

    try:
        urlretrieve(url, target)
    except (HTTPError, URLError) as e:
        raise RuntimeError(
            f"Failed to download '{file_name}' from GitHub URL: {url}. "
            "Check cfg.github_repo / cfg.github_branch / cfg.github_data_dir, "
            "and ensure internet access is enabled in notebook runtime."
        ) from e

    return target


def resolve_training_data_path() -> str:
    candidates = [
        repo_root / "data" / cfg.data_file_name,
        Path("data") / cfg.data_file_name,
        Path("../data") / cfg.data_file_name,
        Path("/kaggle/working/data") / cfg.data_file_name,
        Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1/data") / cfg.data_file_name,
    ]

    for p in candidates:
        if p.exists():
            return str(p)

    if cfg.use_github_fallback:
        if not cfg.github_repo:
            raise ValueError("cfg.github_repo is empty. Set it as 'owner/repo' to enable GitHub fallback ")

        if RUNTIME == "kaggle":
            download_dir = Path("/kaggle/working/data")
        elif RUNTIME == "colab":
            download_dir = Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1/data")
        else:
            download_dir = repo_root / "data"

        downloaded_path = download_data_from_github(
            file_name=cfg.data_file_name,
            repo=cfg.github_repo,
            branch=cfg.github_branch,
            github_data_dir=cfg.github_data_dir,
            local_data_dir=download_dir,
            force_refresh=cfg.force_refresh_github_data,
        )
        return str(downloaded_path)

    raise FileNotFoundError(
        "Cannot find training data file. Checked local candidates and GitHub fallback is disabled. "
        "Please provide data locally or enable cfg.use_github_fallback with valid GitHub settings."
    )


training_path = resolve_training_data_path()
raw_data = get_jsonl_data(training_path)
all_pairs = build_training_pairs(raw_data)

train_pairs, temp_pairs = train_test_split(
    all_pairs,
    test_size=cfg.test_size,
    random_state=SEED,
    shuffle=True,
    stratify=None,
)
val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=cfg.val_ratio_in_temp,
    random_state=SEED,
    shuffle=True,
    stratify=None,
)

print(f"Repo root: {repo_root}")
print(f"Training data path: {training_path}")
print(f"Output path: {cfg.output_path}")
print(f"Total pairs: {len(all_pairs)}")
print(f"Train/Val/Test: {len(train_pairs)} / {len(val_pairs)} / {len(test_pairs)}")

Repo root: e:\AI\NLP\Legal-question-answer-v1
Training data path: e:\AI\NLP\Legal-question-answer-v1\data\training_data_v1.jsonl
Output path: E:\AI\NLP\Legal-question-answer-v1\data\pretokenized_retriever_v1.pt
Total pairs: 9715
Train/Val/Test: 6800 / 1457 / 1458


In [3]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

from utils.retriever_data_utils import tokenize_multi_negative_pairs

cache = {
    "meta": {
        "model_name": cfg.model_name,
        "max_q_len": cfg.max_q_len,
        "max_c_len": cfg.max_c_len,
        "seed": SEED,
        "schema": "one_positive_many_negatives_per_sample",
    },
    "train": tokenize_multi_negative_pairs(train_pairs, tokenizer, cfg.max_q_len, cfg.max_c_len),
    "val": tokenize_multi_negative_pairs(val_pairs, tokenizer, cfg.max_q_len, cfg.max_c_len),
    "test": tokenize_multi_negative_pairs(test_pairs, tokenizer, cfg.max_q_len, cfg.max_c_len),
}

out_path = Path(cfg.output_path)
out_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(cache, out_path)

print(f"Saved pretokenized cache to: {out_path}")
print(f"Cache keys: {list(cache.keys())}")

Saved pretokenized cache to: E:\AI\NLP\Legal-question-answer-v1\data\pretokenized_retriever_v1.pt
Cache keys: ['meta', 'train', 'val', 'test']


## Reuse In Training Notebook

Chạy notebook này trước để tạo cache `.pt`, sau đó trong notebook train bạn chỉ cần load cache thay vì tokenize lại.

### Cross-platform notes (Local / Colab / Kaggle)

- Local: notebook ưu tiên đọc `data/training_data.jsonl` trong project.
- Colab/Kaggle: nếu không có local data file, notebook sẽ thử tải từ GitHub raw URL.
- Đảm bảo `cfg.github_repo` đúng format `owner/repo` nếu auto-detect không ra.

Ví dụ cấu hình cho Kaggle/Colab trước khi chạy cell load data:

```python
cfg.github_repo = "your-github-username/Legal-question-answer-v1"
cfg.github_branch = "main"
cfg.github_data_dir = "data"
cfg.use_github_fallback = True
```

Các bước tích hợp nhanh trong notebook train:
1. Load cache bằng `torch.load(...)`.
2. Tạo `Dataset` từ tensor đã tokenized.
3. Dùng `DataLoader` như cũ với `collate_fn` đơn giản hoặc bỏ `collate_fn` nếu dataset đã trả tensor theo từng sample.

In [4]:
# # Example loader code for retriever_training_v1.ipynb (reuse from utils/)
# import sys
# from pathlib import Path

# # Ensure repository root is importable when running notebook from retriever/
# repo_root = Path.cwd()
# if (repo_root / "utils").exists() is False:
#     repo_root = repo_root.parent
# if str(repo_root) not in sys.path:
#     sys.path.append(str(repo_root))

# from utils.pretokenized_dataset import (
#     PreTokenizedRetrieverDataset,
#     load_pretokenized_cache,
#     validate_cache_meta,
# )

# cache_path = Path("data/pretokenized_retriever_v1.pt")
# if not cache_path.exists():
#     cache_path = Path("../data/pretokenized_retriever_v1.pt")

# loaded = load_pretokenized_cache(str(cache_path), map_location="cpu")
# print("Loaded cache meta:", loaded["meta"])

# # Optional config guard
# validate_cache_meta(
#     loaded,
#     model_name=cfg.model_name,
#     max_q_len=cfg.max_q_len,
#     max_c_len=cfg.max_c_len,
# )

# train_ds_cached = PreTokenizedRetrieverDataset(loaded["train"])
# val_ds_cached = PreTokenizedRetrieverDataset(loaded["val"])
# test_ds_cached = PreTokenizedRetrieverDataset(loaded["test"])
# print(len(train_ds_cached), len(val_ds_cached), len(test_ds_cached))